# TF Ensemble Grid — сигнал тренда с измерением качества

Переработка `feature_engineering/trend_follower_grid.ipynb`. Что изменено и зачем:

| Было | Стало |
|---|---|
| Юниверс = всё, что случайно лежит в кэше | Явный `TRADING_PAIRS` + data-quality gate + `assert` |
| Один EMA(20/200/500) стек | Ансамбль по лог-сетке скоростей (выбор скорости не делается вовсе) |
| `intensity` в сырых единицах цены | Нормировка на реализованную волатильность + кросс-секционный z-score |
| Нет оценки «тренд ли это» | Ортогональная ось качества: Kaufman ER, R² лог-цены по времени, variance ratio |
| Снимок последнего бара 1m | Подтверждение старшим таймфреймом (ресемпл из того же 1m, без доп. загрузки) |
| 40 «сигналов» = одна ставка на рынок | Бета-нейтрализация к BTC, остаточный моментум |
| Сигнал не проверен ничем | Панель forward-меток, IC, децили, порог против round-trip cost |
| Параметры подобраны глазом | Purged walk-forward, выбор плато (не argmax), PBO-lite |
| — | Опциональный meta-labeling слой поверх триггера |

**Порядок рассуждения зафиксирован: сначала горизонт метки, потом lookback'и.** Все окна индикаторов
выводятся из `HORIZON_BARS` через множители, а не задаются независимо.

---

## Предварительно: данные

Свечи качаются штатными ноутбуками из
[`research_notebooks/data_collection`](https://github.com/hummingbot/quants-lab/tree/main/research_notebooks/data_collection):

* `download_candles_all_pairs.ipynb` — весь USDT-юниверс. Поставьте `INTERVALS = ["1m"]`, `DAYS >= 30`
  (меньше 30 дней не хватит на purged walk-forward: 5 фолдов × горизонт × purge).
* `download_specific_candles.ipynb` — если работаете по короткому белому списку пар.
* `load_candles_from_cache.ipynb` — посмотреть, что уже лежит в `app/data/cache/candles/`.

Старший таймфрейм **не качается отдельно** — он ресемплится из 1m внутри этого ноутбука,
чтобы гарантированно совпадали границы баров и не появилось рассогласование источников.

Данные о funding / OI / базисе здесь сознательно не используются. Единственный поток-подобный
признак — `taker_buy_base_volume`, он приходит внутри самой свечи Binance и не требует
дополнительных источников; его можно выключить через `USE_FLOW = False`.

## 0. Конфигурация

Всё, что ниже, выводится из горизонта удержания. Меняете горизонт — окна едут за ним.

In [ ]:
import warnings
from datetime import timedelta

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from core.data_sources.clob import CLOBDataSource

warnings.filterwarnings("ignore")
load_dotenv()

clob = CLOBDataSource()

CONNECTOR_NAME = "binance_perpetual"
INTERVAL = "1m"

# ---- ШАГ 1: горизонт. Это единственное, что задаётся из экономики стратегии.
# Сколько баров в среднем живёт позиция грида от входа до выхода.
HORIZON_BARS = 60          # 60 баров 1m = 1 час

# ---- ШАГ 2: издержки. Порог сигнала не имеет смысла без них.
TAKER_FEE_BPS = 4.5        # 0.045% тейкер Binance USDⓈ-M
MAKER_FEE_BPS = 1.8        # 0.018% мейкер
EXPECTED_FILLS = 2.0       # среднее число заливок на цикл грида
SLIPPAGE_BPS = 0.5
ROUND_TRIP_COST_BPS = EXPECTED_FILLS * (MAKER_FEE_BPS + TAKER_FEE_BPS) / 2 + SLIPPAGE_BPS

# ---- ШАГ 3: окна выводятся из горизонта, а не подбираются независимо.
H = HORIZON_BARS
CFG = dict(
    horizon=H,
    vol_halflife=2 * H,                 # EWMA-волатильность
    z_window=int(16 * H),               # окно нормировки сигнала
    speed_pairs=[(int(0.25 * H), int(0.75 * H)),
                 (int(0.5 * H), int(1.5 * H)),
                 (int(1.0 * H), int(3.0 * H)),
                 (int(2.0 * H), int(6.0 * H))],
    lookback_multipliers=[0.5, 1.0, 2.0],   # проверяем масштаб, а не отдельные окна
    er_window=2 * H,
    r2_window=2 * H,
    vr_q=max(2, H // 4),
    vr_window=10 * H,
    rank_window=25 * H,
    htf_rule="15min",
    htf_vol_halflife=24,
    htf_speed_pairs=[(4, 12), (8, 24), (16, 48)],
    htf_z_window=200,
    flow_window=H,
    beta_window=24 * H,
    resid_mom_window=2 * H,
)

# ---- ШАГ 4: юниверс задаётся ЯВНО. None = взять всё из кэша, но тогда состав логируется.
TRADING_PAIRS = None       # напр. ["BTC-USDT", "ETH-USDT", "SOL-USDT", ...]
MARKET_PROXY = "BTC-USDT"  # прокси рынка для бета-нейтрализации
# минимум истории = прогрев самого длинного окна + горизонт (а не круглое число)
MIN_BARS = CFG["rank_window"] + CFG["beta_window"] + H
MIN_UNIVERSE_COVERAGE = 0.9   # доля пар, чью историю обязано покрыть общее окно
MAX_STALENESS = timedelta(hours=6)
MAX_GAP_RATIO = 0.02       # доля пропущенных 1m баров

USE_FLOW = True
SAMPLE_STRIDE = max(1, H // 2)   # прореживание панели, чтобы forward-окна меньше перекрывались

print(f"Горизонт: {H} баров {INTERVAL}")
print(f"Round-trip cost: {ROUND_TRIP_COST_BPS:.2f} bps  <- сигнал обязан его перебивать")
print(f"Скорости ансамбля: {CFG['speed_pairs']}")

## 1. Юниверс и качество данных

Главная поправка к исходному ноутбуку: состав выборки перестаёт быть побочным эффектом
содержимого кэша. Пары, не прошедшие проверку, отбрасываются **с явным логом причины** —
молчаливое исчезновение пары из юниверса это способ получить необъяснимый результат.

In [ ]:
clob.load_candles_cache(connector_name=CONNECTOR_NAME, interval=INTERVAL)

raw = {}
rejected = []
for (conn, pair, interval), candles in clob.candles_cache.items():
    if conn != CONNECTOR_NAME or interval != INTERVAL:
        continue
    if TRADING_PAIRS is not None and pair not in TRADING_PAIRS:
        continue
    df = candles.data.sort_index()
    df = df[~df.index.duplicated(keep="first")]

    if len(df) < MIN_BARS:
        rejected.append((pair, f"мало баров: {len(df)} < {MIN_BARS}"))
        continue
    span = df.index.max() - df.index.min()
    expected = span / pd.Timedelta(INTERVAL) + 1
    gap_ratio = 1.0 - len(df) / expected
    if gap_ratio > MAX_GAP_RATIO:
        rejected.append((pair, f"дыры в данных: {gap_ratio:.1%}"))
        continue
    staleness = pd.Timestamp.now(tz="UTC").tz_localize(None) - df.index.max()
    if staleness > MAX_STALENESS:
        rejected.append((pair, f"устаревшие данные: {staleness}"))
        continue
    if (df["close"] <= 0).any() or df["close"].isna().any():
        rejected.append((pair, "некорректные цены"))
        continue
    raw[pair] = df

assert len(raw) > 0, "Кэш пуст. Сначала прогоните data_collection/download_candles_all_pairs.ipynb"
assert MARKET_PROXY in raw, f"{MARKET_PROXY} нужен для бета-нейтрализации, но его нет в кэше"

# Общее окно: НЕ по самой молодой паре, иначе одна новая монета обрежет всю историю.
starts = pd.Series({p: df.index.min() for p, df in raw.items()})
ends = pd.Series({p: df.index.max() for p, df in raw.items()})
start = starts.quantile(MIN_UNIVERSE_COVERAGE)
end = ends.min()

late = sorted(starts[starts > start].index)
for p in late:
    rejected.append((p, f"слишком короткая история относительно юниверса (с {starts[p]})"))
raw = {p: df.loc[start:end] for p, df in raw.items() if p not in late}
assert len(raw) > 0, "После выравнивания окна не осталось пар"
assert MARKET_PROXY in raw, f"{MARKET_PROXY} выбыл при выравнивании общего окна"

print(f"Принято пар: {len(raw)}")
print(f"Общее окно: {start} .. {end}  ({(end - start).days} дней, ~{int((end-start)/pd.Timedelta(INTERVAL))} баров)")
print(f"Отброшено: {len(rejected)}")
for p, why in rejected[:15]:
    print(f"   - {p}: {why}")
print("\nЮниверс:", ", ".join(sorted(raw)[:20]), "..." if len(raw) > 20 else "")

## 2. Примитивы

Всё каузально по построению: любое окно смотрит только назад. Проверка на утечку — в разделе 4.

In [ ]:
def log_returns(close: pd.Series) -> pd.Series:
    return np.log(close).diff()


def realized_vol(close: pd.Series, halflife: int) -> pd.Series:
    """Per-bar EWMA volatility of log returns."""
    r = log_returns(close)
    sigma = r.ewm(halflife=halflife, min_periods=halflife).std()
    return sigma.replace(0.0, np.nan)


def atr_pct(df: pd.DataFrame, halflife: int) -> pd.Series:
    prev_close = df["close"].shift(1)
    tr = pd.concat([
        df["high"] - df["low"],
        (df["high"] - prev_close).abs(),
        (df["low"] - prev_close).abs(),
    ], axis=1).max(axis=1)
    return (tr.ewm(halflife=halflife, min_periods=halflife).mean() / df["close"]).replace(0.0, np.nan)


def rolling_rank_pct(s: pd.Series, window: int) -> pd.Series:
    return s.rolling(window, min_periods=window // 2).rank(pct=True)

## 3. Ось 1 — направление: ансамбль вместо выбора скорости

Вопрос «какая EMA лучше» не задаётся. Сигнал усредняется по лог-сетке скоростей,
каждая нормирована на волатильность и сжата в (-1, 1). Это снижает дисперсию оценки
и убирает главный источник переподгонки — выбор одного «победившего» окна.

In [ ]:
def ema_trend_ensemble(close: pd.Series, sigma: pd.Series,
                       speed_pairs, z_window: int) -> pd.DataFrame:
    """Vol-normalised EMA-crossover ensemble (CTA style).

    Returns a frame with one column per speed pair plus the ensemble mean,
    each squashed into (-1, 1).
    """
    out = {}
    for fast, slow in speed_pairs:
        gap = close.ewm(span=fast, min_periods=fast).mean() - close.ewm(span=slow, min_periods=slow).mean()
        # normalise by the price displacement expected over `slow` bars
        y = gap / (close * sigma * np.sqrt(slow))
        z = y / y.rolling(z_window, min_periods=z_window // 2).std()
        out[f"tr_{fast}_{slow}"] = np.tanh(z)
    frame = pd.DataFrame(out, index=close.index)
    frame["trend_ens"] = frame.mean(axis=1, skipna=False)
    return frame

## 4. Ось 2 — качество тренда

Ортогональная к направлению ось: отвечает не «куда», а «тренд ли это вообще».

* **Kaufman ER** — чистое смещение / длина пути.
* **R²** лог-цены по времени — насколько движение линейно.
* **Variance ratio** VR(q) — >1 трендовость, <1 возврат к среднему.

Три метрики сводятся через rolling-перцентили: сырые значения несопоставимы между парами.

In [ ]:
def efficiency_ratio(close: pd.Series, window: int) -> pd.Series:
    """Kaufman ER: net displacement / path length. 0 = noise, 1 = pure trend."""
    net = (close - close.shift(window)).abs()
    path = close.diff().abs().rolling(window, min_periods=window).sum()
    return (net / path.replace(0.0, np.nan)).clip(0, 1)


def trend_r2(close: pd.Series, window: int) -> pd.Series:
    """R^2 of log-price regressed on time (rolling)."""
    t = pd.Series(np.arange(len(close), dtype=float), index=close.index)
    corr = np.log(close).rolling(window, min_periods=window).corr(t)
    return (corr ** 2).clip(0, 1)


def variance_ratio(close: pd.Series, q: int, window: int) -> pd.Series:
    """VR(q) > 1 => trending / positively autocorrelated, < 1 => mean-reverting."""
    r1 = log_returns(close)
    rq = r1.rolling(q, min_periods=q).sum()
    v1 = r1.rolling(window, min_periods=window).var()
    vq = rq.rolling(window, min_periods=window).var()
    return vq / (q * v1.replace(0.0, np.nan))


def trend_quality(close: pd.Series, er_window: int, r2_window: int,
                  vr_q: int, vr_window: int, rank_window: int) -> pd.DataFrame:
    er = efficiency_ratio(close, er_window)
    r2 = trend_r2(close, r2_window)
    vr = variance_ratio(close, vr_q, vr_window)
    q = pd.DataFrame({
        "er": er,
        "r2": r2,
        "vr": vr,
        "er_rank": rolling_rank_pct(er, rank_window),
        "r2_rank": rolling_rank_pct(r2, rank_window),
        "vr_rank": rolling_rank_pct(vr, rank_window),
    })
    q["quality"] = q[["er_rank", "r2_rank", "vr_rank"]].mean(axis=1, skipna=False)
    return q

## 5. Оси 3–5 — старший ТФ, бета-остаток, тейкер-флоу

**Старший ТФ.** Ресемпл из тех же 1m. Индекс свечей Hummingbot — время **открытия** бара,
поэтому ресемпл идёт с `label="left", closed="left"`, а при выравнивании обратно на 1m
делается `shift(1)`: используется только полностью закрытый HTF-бар.

**Бета-остаток.** Тренд альта на перпах — это в основном бета к BTC. Без нейтрализации
топ-10 сигналов окажутся одной и той же ставкой на рынок.

**Тейкер-флоу.** Единственный не-ценовой признак; берётся из `taker_buy_base_volume`
самой свечи, внешних источников не требует.

In [ ]:
def resample_ohlcv(df: pd.DataFrame, rule: str) -> pd.DataFrame:
    """Assumes the index is the bar OPEN time (Hummingbot convention)."""
    agg = {"open": "first", "high": "max", "low": "min", "close": "last"}
    if "volume" in df.columns:
        agg["volume"] = "sum"
    if "taker_buy_base_volume" in df.columns:
        agg["taker_buy_base_volume"] = "sum"
    return df.resample(rule, label="left", closed="left").agg(agg).dropna(subset=["close"])


def align_htf(htf: pd.Series | pd.DataFrame, target_index: pd.Index):
    """Shift by one HTF bar (so only *completed* bars are used) then forward-fill."""
    return htf.shift(1).reindex(target_index, method="ffill")

In [ ]:
def beta_residual_momentum(r: pd.Series, r_mkt: pd.Series,
                           beta_window: int, mom_window: int) -> pd.DataFrame:
    cov = r.rolling(beta_window, min_periods=beta_window // 2).cov(r_mkt)
    var = r_mkt.rolling(beta_window, min_periods=beta_window // 2).var()
    beta = cov / var.replace(0.0, np.nan)
    resid = r - beta * r_mkt
    mom = resid.rolling(mom_window, min_periods=mom_window).sum()
    vol = resid.rolling(mom_window, min_periods=mom_window).std() * np.sqrt(mom_window)
    return pd.DataFrame({"beta": beta, "resid_mom": np.tanh(mom / vol.replace(0.0, np.nan))})

In [ ]:
def taker_imbalance(df: pd.DataFrame, window: int, rank_window: int) -> pd.Series:
    """Signed taker flow from the candle payload itself (no OI / funding needed)."""
    if "taker_buy_base_volume" not in df.columns or "volume" not in df.columns:
        return pd.Series(np.nan, index=df.index)
    buy = df["taker_buy_base_volume"].rolling(window, min_periods=window).sum()
    tot = df["volume"].rolling(window, min_periods=window).sum().replace(0.0, np.nan)
    imb = 2.0 * (buy / tot) - 1.0
    return 2.0 * rolling_rank_pct(imb, rank_window) - 1.0

## 6. Сборка признаков и метка

Вход — по `open` **следующего** бара (сигнал известен только на закрытии текущего),
выход — по `open` через `horizon` баров. Close-to-close дал бы систематически завышенный
результат на 1m.

`fwd_ret_norm` (нормированная на σ·√h) используется для IC, сырая `fwd_ret` в bps —
для сравнения с издержками.

In [ ]:
def build_features(df: pd.DataFrame, cfg: dict, mkt_logret: pd.Series | None = None) -> pd.DataFrame:
    h = cfg["horizon"]
    sigma = realized_vol(df["close"], halflife=cfg["vol_halflife"])
    out = pd.DataFrame(index=df.index)
    out["close"] = df["close"]
    out["sigma"] = sigma
    out["atr_pct"] = atr_pct(df, halflife=cfg["vol_halflife"])

    for mult in cfg["lookback_multipliers"]:
        pairs = [(max(2, int(round(f * mult))), max(3, int(round(s * mult))))
                 for f, s in cfg["speed_pairs"]]
        ens = ema_trend_ensemble(df["close"], sigma, pairs, z_window=cfg["z_window"])
        out[f"trend_x{mult:g}"] = ens["trend_ens"]

    q = trend_quality(df["close"], er_window=cfg["er_window"], r2_window=cfg["r2_window"],
                      vr_q=cfg["vr_q"], vr_window=cfg["vr_window"], rank_window=cfg["rank_window"])
    out["quality"] = q["quality"]
    out["er"] = q["er"]
    out["vr"] = q["vr"]

    htf = resample_ohlcv(df, cfg["htf_rule"])
    htf_sigma = realized_vol(htf["close"], halflife=cfg["htf_vol_halflife"])
    htf_ens = ema_trend_ensemble(htf["close"], htf_sigma, cfg["htf_speed_pairs"],
                                 z_window=cfg["htf_z_window"])["trend_ens"]
    out["trend_htf"] = align_htf(htf_ens, df.index)

    out["flow"] = taker_imbalance(df, window=cfg["flow_window"], rank_window=cfg["rank_window"])

    if mkt_logret is not None:
        r = log_returns(df["close"])
        bm = beta_residual_momentum(r, mkt_logret.reindex(df.index),
                                    beta_window=cfg["beta_window"], mom_window=cfg["resid_mom_window"])
        out["beta"] = bm["beta"]
        out["resid_mom"] = bm["resid_mom"]
    else:
        out["beta"] = np.nan
        out["resid_mom"] = np.nan

    # forward label: enter at next bar's open, exit h bars later
    entry = df["open"].shift(-1)
    exit_ = df["open"].shift(-(1 + h))
    out["fwd_ret"] = exit_ / entry - 1.0
    out["fwd_ret_norm"] = out["fwd_ret"] / (sigma * np.sqrt(h)).replace(0.0, np.nan)
    return out


def combine_score(f: pd.DataFrame, mult: float, w_flow: float = 0.0, w_resid: float = 0.0) -> pd.Series:
    base = f[f"trend_x{mult:g}"]
    if w_flow or w_resid:
        base = (1 - w_flow - w_resid) * base + w_flow * f["flow"].fillna(0.0) + w_resid * f["resid_mom"].fillna(0.0)
    return base.clip(-1, 1)

In [ ]:
mkt_logret = log_returns(raw[MARKET_PROXY]["close"])

features = {}
errors = []
for pair, df in raw.items():
    try:
        f = build_features(df, CFG, mkt_logret=mkt_logret)
        if not USE_FLOW:
            f["flow"] = np.nan
        features[pair] = f
    except Exception as e:
        errors.append((pair, f"{type(e).__name__}: {e}"))

print(f"Признаки посчитаны для {len(features)} пар, ошибок: {len(errors)}")
for p, e in errors[:10]:
    print("  ", p, e)

example = features[MARKET_PROXY]
display(example[["trend_x1", "quality", "trend_htf", "flow", "resid_mom", "fwd_ret_norm"]]
        .describe().T[["count", "mean", "std", "min", "max"]])

### Панель

Прореживание с шагом `SAMPLE_STRIDE` не убирает перекрытие forward-окон полностью,
но снижает автокорреляцию наблюдений. Это важно: без него t-статистики
будут завышены в разы.

In [ ]:
panel_rows = []
keep_cols = ["close", "sigma", "atr_pct", "quality", "er", "vr", "trend_htf",
             "flow", "beta", "resid_mom", "fwd_ret", "fwd_ret_norm"]
keep_cols += [f"trend_x{m:g}" for m in CFG["lookback_multipliers"]]

for pair, f in features.items():
    s = f.iloc[::SAMPLE_STRIDE][keep_cols].copy()
    s["trading_pair"] = pair
    panel_rows.append(s)

panel = pd.concat(panel_rows)
panel.index.name = "timestamp"
panel = panel.set_index("trading_pair", append=True)
panel = panel.dropna(subset=["trend_x1", "quality", "fwd_ret", "fwd_ret_norm"])

# кросс-секционная нормировка: сравниваем пары между собой на одном срезе времени
for m in CFG["lookback_multipliers"]:
    col = f"trend_x{m:g}"
    g = panel.groupby(level="timestamp")[col]
    panel[f"{col}_cs"] = ((g.rank(pct=True) - 0.5) * 2.0)

print("Панель:", panel.shape)
print("Срезов времени:", panel.index.get_level_values("timestamp").nunique())
print("Пар в срезе (медиана):", int(panel.groupby(level="timestamp").size().median()))
display(panel.head(3))

## 7. Проверка на утечку

Дешёвый и обязательный тест: пересчитать признаки на усечённой истории и сравнить
с полной на общем хвосте. Любое ненулевое расхождение = будущее просочилось в прошлое,
и все метрики ниже недействительны.

In [ ]:
probe_pair = MARKET_PROXY
df_full = raw[probe_pair]
cut = int(len(df_full) * 0.7)

f_full = build_features(df_full, CFG, mkt_logret=mkt_logret)
f_trunc = build_features(df_full.iloc[:cut], CFG, mkt_logret=mkt_logret.iloc[:cut])

check_cols = ["sigma", "atr_pct", "trend_x1", "quality", "trend_htf", "flow", "resid_mom"]
last = f_trunc.index[-1]
tail = 500
delta = (f_full.loc[:last, check_cols].tail(tail) - f_trunc.loc[:last, check_cols].tail(tail)).abs().max()

print("max |полная - усечённая| на общем хвосте:")
display(delta.to_frame("max_abs_diff"))
assert (delta.fillna(0) < 1e-9).all(), "УТЕЧКА: признак зависит от будущих баров"
print("\nУтечки нет.")

## 8. Оценка сигнала

Три вопроса, в этом порядке:

1. **Ортогональны ли оси** — или мы посчитали один фактор четырьмя способами.
2. **Есть ли IC** — кросс-секционный ранговый Spearman по срезам, с t-статистикой.
3. **Перебивает ли издержки** — gross bps против `ROUND_TRIP_COST_BPS`.

Оценивается качество *сигнала* (markout по forward-доходности), а не PnL всего грида.
Иначе качество сигнала и качество исполнения смешаются.

In [ ]:
def cross_sectional_ic(panel: pd.DataFrame, score_col: str, ret_col: str = "fwd_ret_norm") -> pd.Series:
    def _ic(g):
        if len(g) < 5:
            return np.nan
        a = g[score_col].rank()
        b = g[ret_col].rank()
        if a.std() == 0 or b.std() == 0:
            return np.nan
        return a.corr(b)
    return panel.groupby(level="timestamp", group_keys=False).apply(_ic)


def ic_summary(ic: pd.Series) -> dict:
    ic = ic.dropna()
    if len(ic) < 3:
        return {"n": len(ic), "mean_ic": np.nan, "t_stat": np.nan}
    return {"n": int(len(ic)), "mean_ic": float(ic.mean()),
            "t_stat": float(ic.mean() / ic.std(ddof=1) * np.sqrt(len(ic)))}


def threshold_report(panel: pd.DataFrame, score_col: str, thresholds,
                     quality_cut: float, cost_bps: float) -> pd.DataFrame:
    rows = []
    q = panel["quality"]
    for th in thresholds:
        sel = panel[(panel[score_col].abs() >= th) & (q >= quality_cut)]
        if len(sel) == 0:
            rows.append({"threshold": th, "n": 0, "gross_bps": np.nan,
                         "net_bps": np.nan, "hit_rate": np.nan})
            continue
        signed = np.sign(sel[score_col]) * sel["fwd_ret"]
        gross = signed.mean() * 1e4
        rows.append({
            "threshold": th,
            "n": int(len(sel)),
            "gross_bps": gross,
            "net_bps": gross - cost_bps,
            "hit_rate": float((signed > 0).mean()),
        })
    return pd.DataFrame(rows)


def purged_folds(times: pd.DatetimeIndex, n_splits: int, purge: pd.Timedelta):
    """Contiguous test blocks; train excludes a purge window on both sides."""
    times = pd.DatetimeIndex(sorted(pd.unique(times)))
    edges = np.linspace(0, len(times), n_splits + 1).astype(int)
    folds = []
    for i in range(n_splits):
        lo, hi = times[edges[i]], times[edges[i + 1] - 1]
        test = (times >= lo) & (times <= hi)
        train = ((times < lo - purge) | (times > hi + purge))
        folds.append((times[train], times[test]))
    return folds

In [ ]:
comp_cols = ["trend_x0.5", "trend_x1", "trend_x2", "quality", "trend_htf", "flow", "resid_mom"]
corr = panel[comp_cols].corr(method="spearman")
print("Корреляция компонент (Spearman) — ищем НИЗКИЕ значения вне блока trend_*:")
display(corr.round(2))

In [ ]:
rows = []
for col in ["trend_x0.5", "trend_x1", "trend_x2", "trend_x1_cs", "trend_htf", "flow", "resid_mom"]:
    if panel[col].notna().sum() < 100:
        continue
    s = ic_summary(cross_sectional_ic(panel, col))
    s["signal"] = col
    rows.append(s)
ic_table = pd.DataFrame(rows).set_index("signal")[["n", "mean_ic", "t_stat"]]
print("Кросс-секционный IC против нормированной forward-доходности:")
display(ic_table.round(4))
print("Ориентир: |t| > 3 при прореженной панели — минимум, ниже читать как шум.")

In [ ]:
print(f"Издержки round-trip: {ROUND_TRIP_COST_BPS:.2f} bps\n")
for qcut in [0.0, 0.5, 0.7]:
    rep = threshold_report(panel, "trend_x1", [0.0, 0.2, 0.4, 0.6, 0.8],
                           quality_cut=qcut, cost_bps=ROUND_TRIP_COST_BPS)
    rep.insert(0, "quality_cut", qcut)
    display(rep.round(3))

In [ ]:
# Децили: монотонность важнее среднего значения
d = panel.copy()
d["decile"] = pd.qcut(d["trend_x1_cs"], 10, labels=False, duplicates="drop")
tbl = d.groupby("decile").agg(
    n=("fwd_ret", "size"),
    mean_fwd_bps=("fwd_ret", lambda x: x.mean() * 1e4),
    mean_fwd_norm=("fwd_ret_norm", "mean"),
    hit=("fwd_ret", lambda x: (x > 0).mean()),
)
display(tbl.round(3))
print("Монотонный профиль по децилям >> высокий средний bps в одном крайнем дециле.")

## 9. Purged walk-forward и выбор плато

Здесь оптимизируется **не сигнал, а решающее правило поверх него**: масштаб окон,
порог |score| и отсечка по качеству. Три принципа:

* **Purge + embargo** вокруг тестового блока — иначе перекрывающиеся forward-окна
  протащат тестовую информацию в train.
* **Плато, не argmax.** Берётся центр устойчивой области после сглаживания поверхности.
  Максимум in-sample при переборе десятков конфигураций не несёт информации.
* **PBO-lite.** Для каждого фолда смотрим, в какой перцентиль OOS-распределения попала
  конфигурация, выбранная на train. Систематически низкий ранг = переподгонка.

In [ ]:
def evaluate_config(panel: pd.DataFrame, mult: float, th: float, qcut: float,
                    cost_bps: float, min_obs: int = 50) -> dict:
    col = f"trend_x{mult:g}"
    sel = panel[(panel[col].abs() >= th) & (panel["quality"] >= qcut)]
    if len(sel) < min_obs:
        return {"n": int(len(sel)), "net_bps": np.nan, "hit_rate": np.nan}
    signed = np.sign(sel[col]) * sel["fwd_ret"]
    return {"n": int(len(sel)), "net_bps": float(signed.mean() * 1e4 - cost_bps),
            "hit_rate": float((signed > 0).mean())}


def grid_search(panel: pd.DataFrame, mults, thresholds, qcuts, cost_bps: float) -> pd.DataFrame:
    rows = []
    for m in mults:
        ic_m = ic_summary(cross_sectional_ic(panel, f"trend_x{m:g}"))["mean_ic"]
        for th in thresholds:
            for qc in qcuts:
                r = evaluate_config(panel, m, th, qc, cost_bps)
                r.update({"mult": m, "threshold": th, "quality_cut": qc, "ic": ic_m})
                rows.append(r)
    return pd.DataFrame(rows)


def pick_plateau(grid: pd.DataFrame, metric: str = "net_bps") -> pd.Series:
    """Pick the centre of the most robust region, not the argmax."""
    g = grid.dropna(subset=[metric]).copy()
    if g.empty:
        return pd.Series(dtype=float)
    piv = g.pivot_table(index=["mult", "threshold"], columns="quality_cut", values=metric)
    smoothed = piv.rolling(3, center=True, min_periods=2).mean()
    smoothed = smoothed.T.rolling(3, center=True, min_periods=2).mean().T
    try:
        flat = smoothed.stack(future_stack=True).dropna()
    except TypeError:  # pandas < 2.1
        flat = smoothed.stack().dropna()
    if flat.empty:
        return g.sort_values(metric, ascending=False).iloc[0]
    (mult, th), qc = flat.idxmax()[:2], flat.idxmax()[2]
    hit = g[(g["mult"] == mult) & (g["threshold"] == th) & (g["quality_cut"] == qc)]
    return hit.iloc[0] if len(hit) else g.sort_values(metric, ascending=False).iloc[0]


def walk_forward(panel: pd.DataFrame, mults, thresholds, qcuts, cost_bps: float,
                 n_splits: int, purge: pd.Timedelta) -> pd.DataFrame:
    times = panel.index.get_level_values("timestamp")
    rows = []
    for k, (tr_times, te_times) in enumerate(purged_folds(times, n_splits, purge)):
        tr = panel[times.isin(tr_times)]
        te = panel[times.isin(te_times)]
        if len(tr) < 200 or len(te) < 100:
            continue
        g = grid_search(tr, mults, thresholds, qcuts, cost_bps)
        best = pick_plateau(g)
        if best.empty:
            continue
        oos = evaluate_config(te, best["mult"], best["threshold"], best["quality_cut"], cost_bps)
        # PBO-lite: where does the IS-best config rank out of sample?
        g_oos = grid_search(te, mults, thresholds, qcuts, cost_bps).dropna(subset=["net_bps"])
        rank = float((g_oos["net_bps"] < oos["net_bps"]).mean()) if len(g_oos) else np.nan
        rows.append({"fold": k, "mult": best["mult"], "threshold": best["threshold"],
                     "quality_cut": best["quality_cut"], "is_net_bps": best["net_bps"],
                     "oos_net_bps": oos["net_bps"], "oos_n": oos["n"],
                     "oos_hit_rate": oos["hit_rate"], "oos_rank_pct": rank})
    return pd.DataFrame(rows)

In [ ]:
MULTS = CFG["lookback_multipliers"]
THRESHOLDS = [0.0, 0.2, 0.4, 0.6]
QCUTS = [0.0, 0.3, 0.5, 0.7]
N_SPLITS = 5
PURGE = pd.Timedelta(INTERVAL) * HORIZON_BARS * 3   # горизонт + эмбарго

n_configs = len(MULTS) * len(THRESHOLDS) * len(QCUTS)
wf = walk_forward(panel, MULTS, THRESHOLDS, QCUTS, ROUND_TRIP_COST_BPS, N_SPLITS, PURGE)

if wf.empty:
    print("Недостаточно данных для walk-forward. Качайте больше дней в data_collection.")
else:
    display(wf.round(3))
    oos = wf["oos_net_bps"].dropna()
    print(f"\nКонфигураций перебрано: {n_configs} (учитывайте это при чтении t-статистики)")
    print(f"OOS net: медиана {oos.median():.2f} bps, фолдов с net>0: {(oos > 0).sum()}/{len(oos)}")
    print(f"Средний IS->OOS спад: {(wf['is_net_bps'] - wf['oos_net_bps']).mean():.2f} bps")
    print(f"PBO-lite (средний OOS-перцентиль выбранной конфигурации): {wf['oos_rank_pct'].mean():.2f}")
    print("   < 0.5 систематически => правило подогнано под train, а не под рынок.")

In [ ]:
if not wf.empty and wf["oos_net_bps"].notna().any():
    PROD_PARAMS = {
        "mult": float(wf["mult"].median()),
        "threshold": float(wf["threshold"].median()),
        "quality_cut": float(wf["quality_cut"].median()),
    }
    PROD_OOS_NET_BPS = float(wf["oos_net_bps"].median())
else:
    PROD_PARAMS, PROD_OOS_NET_BPS = {"mult": 1.0, "threshold": 0.4, "quality_cut": 0.5}, float("nan")

if PROD_PARAMS["mult"] not in CFG["lookback_multipliers"]:
    PROD_PARAMS["mult"] = min(CFG["lookback_multipliers"],
                              key=lambda m: abs(m - PROD_PARAMS["mult"]))

print("Параметры для прода (медиана по фолдам, не argmax):", PROD_PARAMS)
print(f"Ожидаемый OOS net edge: {PROD_OOS_NET_BPS:.2f} bps за {HORIZON_BARS} баров")

## 10. Meta-labeling (опционально)

Первичный триггер остаётся тупым и не оптимизируется. Поверх него учится классификатор
«брать / не брать» на признаках режима: качество тренда, согласие со старшим ТФ,
волатильность, флоу, час суток. Это сэмпл-эффективнее, чем перебор параметров самого триггера.

Метка положительная, если сделка в направлении сигнала перекрыла издержки.
Валидация — те же purged-фолды.

In [ ]:
META_FEATURES = ["score", "score_abs", "quality", "er", "vr", "htf_agree",
                 "flow_agree", "resid_agree", "atr_pct", "hour"]


def build_meta_dataset(panel: pd.DataFrame, score_col: str, cost_bps: float):
    d = panel[panel[score_col].abs() > 0].copy()
    direction = np.sign(d[score_col])
    signed_bps = direction * d["fwd_ret"] * 1e4
    y = (signed_bps > cost_bps).astype(int)
    ts = d.index.get_level_values("timestamp")
    X = pd.DataFrame({
        "score": d[score_col],
        "score_abs": d[score_col].abs(),
        "quality": d["quality"],
        "er": d["er"],
        "vr": d["vr"],
        "htf_agree": np.sign(d["trend_htf"]).fillna(0.0) * direction,
        "flow_agree": d["flow"].fillna(0.0) * direction,
        "resid_agree": d["resid_mom"].fillna(0.0) * direction,
        "atr_pct": d["atr_pct"],
        "hour": ts.hour.astype(float),
    }, index=d.index)
    return X, y, signed_bps


def meta_cv(X, y, signed_bps, n_splits: int, purge: pd.Timedelta, top_q: float = 0.7):
    from sklearn.ensemble import HistGradientBoostingClassifier
    times = X.index.get_level_values("timestamp")
    rows = []
    for k, (tr_t, te_t) in enumerate(purged_folds(times, n_splits, purge)):
        tr, te = times.isin(tr_t), times.isin(te_t)
        if tr.sum() < 500 or te.sum() < 200:
            continue
        clf = HistGradientBoostingClassifier(max_depth=3, max_iter=200,
                                             learning_rate=0.05, l2_regularization=1.0)
        clf.fit(X[tr], y[tr])
        p = pd.Series(clf.predict_proba(X[te])[:, 1], index=X.index[te])
        cut = p.quantile(top_q)
        keep = p >= cut
        rows.append({
            "fold": k,
            "base_rate": float(y[te].mean()),
            "meta_precision": float(y[te][keep.values].mean()),
            "base_net_bps": float(signed_bps[te].mean()),
            "meta_net_bps": float(signed_bps[te][keep.values].mean()),
            "kept": int(keep.sum()),
            "total": int(te.sum()),
        })
    return pd.DataFrame(rows)

In [ ]:
score_col = f"trend_x{PROD_PARAMS['mult']:g}"
X, y, signed_bps = build_meta_dataset(panel, score_col, ROUND_TRIP_COST_BPS)
print("Датасет:", X.shape, "| базовая доля прибыльных:", round(float(y.mean()), 3))

try:
    meta = meta_cv(X, y, signed_bps, n_splits=N_SPLITS, purge=PURGE, top_q=0.7)
    display(meta.round(3))
    if not meta.empty:
        lift = (meta["meta_precision"] - meta["base_rate"]).mean()
        bps_lift = (meta["meta_net_bps"] - meta["base_net_bps"]).mean()
        print(f"\nСредний прирост precision: {lift:+.3f}")
        print(f"Средний прирост net: {bps_lift:+.2f} bps")
        print("Прирост меньше 0.02 precision — слой не нужен, это шум.")
except ImportError:
    print("scikit-learn не установлен — раздел пропущен.")

## 11. Продакшн-сигнал

Сигнал выпускается **только если** walk-forward показал положительный OOS net edge.
Это защита от главного режима отказа: красивый ноутбук выкатывает в прод отрицательное
матожидание, потому что порог сигнала не был привязан к издержкам.

In [ ]:
from core.features import FeatureStorage
from core.features.models import Feature, Signal

SAVE_TO_MONGO = False
GRID_RANGE_MULT = 1.0
FEATURE_NAME = "tf_ensemble"

last_rows = []
for pair, f in features.items():
    row = f.dropna(subset=[score_col, "quality", "atr_pct"]).tail(1)
    if row.empty:
        continue
    r = row.iloc[0]
    last_rows.append({
        "trading_pair": pair,
        "timestamp": row.index[0],
        "score": float(r[score_col]),
        "quality": float(r["quality"]),
        "trend_htf": float(r["trend_htf"]) if pd.notna(r["trend_htf"]) else 0.0,
        "resid_mom": float(r["resid_mom"]) if pd.notna(r["resid_mom"]) else 0.0,
        "flow": float(r["flow"]) if pd.notna(r["flow"]) else 0.0,
        "beta": float(r["beta"]) if pd.notna(r["beta"]) else np.nan,
        "price": float(r["close"]),
        "atr_pct": float(r["atr_pct"]),
        "sigma": float(r["sigma"]),
    })

live = pd.DataFrame(last_rows)
assert not live.empty, "Нет свежих строк — проверьте актуальность кэша"

live["score_cs"] = (live["score"].rank(pct=True) - 0.5) * 2.0
live["htf_agree"] = np.sign(live["trend_htf"]) == np.sign(live["score"])
live["passes"] = (
    (live["score"].abs() >= PROD_PARAMS["threshold"]) &
    (live["quality"] >= PROD_PARAMS["quality_cut"]) &
    live["htf_agree"]
)
live = live.sort_values("score", key=abs, ascending=False)

print(f"Проверено пар: {len(live)} | прошло фильтр: {int(live['passes'].sum())}")
display(live[["trading_pair", "score", "score_cs", "quality", "trend_htf",
              "resid_mom", "beta", "atr_pct", "passes"]].head(15).round(4))

In [ ]:
edge_ok = np.isfinite(PROD_OOS_NET_BPS) and PROD_OOS_NET_BPS > 0

all_features, all_signals, grid_plan = [], [], []
for _, r in live.iterrows():
    feat_value = {
        "score": r["score"], "score_cs": r["score_cs"], "quality": r["quality"],
        "trend_htf": r["trend_htf"], "resid_mom": r["resid_mom"], "flow": r["flow"],
        "beta": r["beta"] if np.isfinite(r["beta"]) else 0.0,
        "atr_pct": r["atr_pct"], "sigma": r["sigma"], "price": r["price"],
    }
    all_features.append(Feature(
        feature_name=FEATURE_NAME,
        trading_pair=r["trading_pair"],
        connector_name=CONNECTOR_NAME,
        value=feat_value,
        info={"interval": INTERVAL, "horizon_bars": HORIZON_BARS,
              "params": PROD_PARAMS, "cost_bps": ROUND_TRIP_COST_BPS,
              "oos_net_bps": PROD_OOS_NET_BPS,
              "speed_pairs": [list(p) for p in CFG["speed_pairs"]]},
    ))

    if not (r["passes"] and edge_ok):
        continue

    direction = 1 if r["score"] > 0 else -1
    all_signals.append(Signal(
        signal_name=f"{FEATURE_NAME}_h{HORIZON_BARS}",
        trading_pair=r["trading_pair"],
        category="tf",
        value=float(np.clip(r["score"], -1, 1)),
    ))

    # диапазон грида — из ATR на горизонте, а не из max-min окна
    range_pct = float(r["atr_pct"] * np.sqrt(HORIZON_BARS) * GRID_RANGE_MULT)
    price = r["price"]
    if direction > 0:
        start, end, limit = price * (1 - 0.5 * range_pct), price * (1 + 1.5 * range_pct), price * (1 - 0.7 * range_pct)
    else:
        start, end, limit = price * (1 + 0.5 * range_pct), price * (1 - 1.5 * range_pct), price * (1 + 0.7 * range_pct)
    grid_plan.append({
        "trading_pair": r["trading_pair"], "side": "LONG" if direction > 0 else "SHORT",
        "price": price, "range_pct": range_pct,
        "start_price": start, "end_price": end, "limit_price": limit,
    })

print(f"Features: {len(all_features)} | Signals: {len(all_signals)}")
if not edge_ok:
    print("!! OOS net edge <= 0 или не определён — сигналы не выпускаются. Features сохраняются как измерение.")
if grid_plan:
    display(pd.DataFrame(grid_plan).round(6))

In [ ]:
if SAVE_TO_MONGO:
    storage = FeatureStorage()
    await storage.connect()
    await storage.save_features(all_features)
    if all_signals:
        await storage.save_signals(all_signals)
    print(f"Сохранено: {len(all_features)} features, {len(all_signals)} signals")
else:
    print("SAVE_TO_MONGO = False — запись пропущена")

### Визуальная проверка топ-пары

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

top_pair = live.iloc[0]["trading_pair"]
f = features[top_pair].tail(3000)

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.5, 0.25, 0.25],
                    vertical_spacing=0.03,
                    subplot_titles=(f"{top_pair} — цена", "score / HTF", "quality / ATR%"))
fig.add_trace(go.Scatter(x=f.index, y=f["close"], name="close", line=dict(width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=f.index, y=f[score_col], name="score", line=dict(width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=f.index, y=f["trend_htf"], name="HTF", line=dict(width=1, dash="dot")), row=2, col=1)
fig.add_hline(y=PROD_PARAMS["threshold"], line=dict(dash="dash", width=1), row=2, col=1)
fig.add_hline(y=-PROD_PARAMS["threshold"], line=dict(dash="dash", width=1), row=2, col=1)
fig.add_trace(go.Scatter(x=f.index, y=f["quality"], name="quality", line=dict(width=1)), row=3, col=1)
fig.add_trace(go.Scatter(x=f.index, y=f["atr_pct"], name="ATR%", line=dict(width=1), yaxis="y4"), row=3, col=1)
fig.add_hline(y=PROD_PARAMS["quality_cut"], line=dict(dash="dash", width=1), row=3, col=1)
fig.update_layout(height=850, width=1400, showlegend=True, title=f"{top_pair}: компоненты сигнала")
fig.show()

## 12. Что этот ноутбук НЕ доказывает

Читать перед тем, как ставить в `config/tf_pipeline.yml`.

1. **Это markout сигнала, а не PnL стратегии.** Измерена forward-доходность от `open`
   следующего бара. Реальный грид входит частями, по лимиткам, с частичным неисполнением
   и adverse selection на заливках. Разрыв между markout и реализованным PnL — отдельная работа.
2. **Издержки — константа.** `ROUND_TRIP_COST_BPS` — грубая модель. Фактические комиссии
   стоит проверять через `/fapi/v1/commissionRate` и `/fapi/v1/userTrades`, а не брать из конфига.
3. **Выживаемость юниверса.** В кэше лежат пары, которые торгуются *сегодня*. Делистнутые
   отсутствуют, что смещает результат вверх на любом окне длиннее пары месяцев.
4. **Перекрытие forward-окон.** `SAMPLE_STRIDE` смягчает, но не устраняет. t-статистики
   остаются оптимистичными; относитесь к ним как к порядку величины.
5. **Число перебранных конфигураций.** Перебрано `len(MULTS)*len(THRESHOLDS)*len(QCUTS)`
   вариантов на каждом фолде. Без поправки на множественное тестирование максимальный
   in-sample результат не является оценкой.
6. **Один режим рынка.** 30–60 дней 1m это, как правило, один режим. Положительный OOS
   на таком окне не переносится через смену режима автоматически.

### Разумный следующий шаг

Прогнать вариант с `HORIZON_BARS` = 15 / 60 / 240 и посмотреть, где IC максимален.
Если оптимальный горизонт не совпадает со временем жизни позиции в гриде — чинить надо
не сигнал, а стратегию исполнения.